In [1]:
## LIBRARIES ----------------------------------------------------- #
#%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive

## INPUT ------------------------------------------------------------ #
W = 500
Z = 3.2
kappa = [2,3,4,5,6]
deform = 0.00

## SETUP ------------------------------------------------------------ #
ptsize = 25
area = W*W*np.sqrt(3)*0.5*(1+deform)
colours = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [2]:
## LOAD DATA ------------------------------------------------- #
for count, kap in enumerate(kappa):
    data1 = np.loadtxt(f'stress/W{W}_Z{Z:.2f}_K{kap}_E{deform:.2f}_V0_r1.dat')
    data2 = np.loadtxt(f'stress/W{W}_Z{Z:.2f}_K{kap}_E{deform:.2f}_V0_r-1.dat')
    data3 = np.loadtxt(f'stress/W{W}_Z{Z:.2f}_K{kap}_E{deform:.2f}_V1_r1.dat')
    data4 = np.loadtxt(f'stress/W{W}_Z{Z:.2f}_K{kap}_E{deform:.2f}_V1_r-1.dat')

    strain = abs(data3[:,0])
    #energy = np.amin([data1[:,1], data2[:,1], data3[:,1], data4[:,1]], axis=0) / area
    energy = 0.25*(data1[:,1] + data2[:,1] + data3[:,1] + data4[:,1]) / area

    stress = np.diff(energy) / np.diff(strain)
    if (kap==2): K2 = np.diff(stress) / np.diff(strain[:-1])
    elif (kap==3): K3 = np.diff(stress) / np.diff(strain[:-1])
    elif (kap==4): K4 = np.diff(stress) / np.diff(strain[:-1])    
    elif (kap==5): K5 = np.diff(stress) / np.diff(strain[:-1])    
    elif (kap==6): K6 = np.diff(stress) / np.diff(strain[:-1])
    elif (kap==0): K0 = np.diff(stress) / np.diff(strain[:-1])

In [3]:
## STRESS VS STRAIN -------------------------------- #
def update_ss( critStrain ):
    plt.figure()

    plt.scatter(strain[:-2], K2, s=ptsize, label='1e-2', c=colours[0])
    plt.scatter(strain[:-2], K3, s=ptsize, label='1e-3', c=colours[1])
    plt.scatter(strain[:-2], K4, s=ptsize, label='1e-4', c=colours[2])
    plt.scatter(strain[:-2], K5, s=ptsize, label='1e-5', c=colours[3])
    plt.scatter(strain[:-2], K6, s=ptsize, label='1e-6', c=colours[4])
    #plt.scatter(strain[:-2], K0, s=ptsize, label='0', c=colours[5])

    critStrain_sug = strain[np.argmax(np.diff(K6)/np.diff(strain[:-2]))]
    #critStrain = strain[np.amin(np.where(K0>0))]

    print( f"Critical Strain = {critStrain_sug}" )
    plt.axvline( x = critStrain, c='k', ls='--' )

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel(r'$\gamma$', fontsize=15)
    plt.ylabel('K', fontsize=15)
    plt.legend(loc='lower right')

In [4]:
## STRESS-STRAIN PLOT WITH SLIDERS -------------------------------- #
interactive_plot_SS = interactive( update_ss,
    critStrain = widgets.FloatSlider(min=0.01, max=0.2, step=1e-3, value=0.01, readout_format='.3f', description="gamC"))

display(interactive_plot_SS)

interactive(children=(FloatSlider(value=0.01, description='gamC', max=0.2, min=0.01, readout_format='.3f', ste…

In [5]:
## FUNCTION TO UPDATE COLLAPSE PLOT ------------------------------------- #
def update_collapse(critStrain, f, begin, end, a, b, c):
    plt.figure()
    phi = f+1.5

    modDelGam = abs(strain[begin:-end] - critStrain)[:-2]
    dgf = modDelGam**-f
    dgp = modDelGam**-phi

    plt.scatter(dgp*(10**-2), K2[begin:-end]*dgf, s=ptsize, label='1e-2', c=colours[0])
    plt.scatter(dgp*(10**-3), K3[begin:-end]*dgf, s=ptsize, label='1e-3', c=colours[1])
    plt.scatter(dgp*(10**-4), K4[begin:-end]*dgf, s=ptsize, label='1e-4', c=colours[2])
    plt.scatter(dgp*(10**-5), K5[begin:-end]*dgf, s=ptsize, label='1e-5', c=colours[3])
    plt.scatter(dgp*(10**-6), K6[begin:-end]*dgf, s=ptsize, label='1e-6', c=colours[4])

    plt.plot([10**-6, 1], [10**-a, 10**-a], 'k--')
    plt.plot([10**-3, 10**-0], [10**(-4-b), 10**(-1-b)], 'k--')
    plt.plot([10**1, 10**4], [10**(1*(f/phi)-c), 10**(4*(f/phi)-c)], 'k--')

    plt.xlabel(r'$\kappa|\Delta\gamma|^{-\phi}$', fontsize=15)
    plt.ylabel(r'K$|\Delta\gamma|^{-f}$', fontsize=15)
    plt.xscale('log')
    plt.yscale('log')

In [7]:
## COLLAPSE PLOT WITH SLIDERS ----------------------------------- #
critStrain = 0.105
interactive_plot_col = interactive( update_collapse,
    critStrain = widgets.FloatSlider(min=critStrain-0.05, max=critStrain+0.05, step=1e-3, value=critStrain, readout_format='.3f', description="gamC"),
    f = widgets.FloatSlider(min=0.6, max=0.75, step=0.01, value=0.7, description="f"),
    #phi = widgets.FloatSlider(min=2.0, max=3.0, step=0.1, value=2.1, description="phi"),
    begin = widgets.IntSlider(min=0, max=50, step=1, value=10, description="begin"),
    end = widgets.IntSlider(min=10, max=30, step=1, value=20, description="end"),
	a = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.4, description="a"),
	b = widgets.FloatSlider(min=0, max=1, step=0.01, value=0.3, description="b"),
	c = widgets.FloatSlider(min=0.0, max=1.5, step=0.1, value=0.8, readout_format='.1f', description="c"))

display(interactive_plot_col)

interactive(children=(FloatSlider(value=0.105, description='gamC', max=0.155, min=0.05499999999999999, readout…

In [15]:
## FUNCTION TO UPDATE F PLOT ------------------------------------- #
def update_f(critStrain, end, a, f):
    plt.figure()

    gamCInd = np.amin(np.where(strain >= critStrain))
    delGam = (strain[gamCInd:-2] - critStrain)

    plt.scatter( delGam[:end], K2[gamCInd:gamCInd+end], s=ptsize, label='1e-2', color=colours[0] )
    plt.scatter( delGam[:end], K3[gamCInd:gamCInd+end], s=ptsize, label='1e-3', color=colours[1] )
    plt.scatter( delGam[:end], K4[gamCInd:gamCInd+end], s=ptsize, label='1e-4', color=colours[2] )
    plt.scatter( delGam[:end], K5[gamCInd:gamCInd+end], s=ptsize, label='1e-5', color=colours[3] )
    plt.scatter( delGam[:end], K6[gamCInd:gamCInd+end], s=ptsize, label='1e-6', color=colours[4] )

    plt.plot([delGam[0], delGam[end]], [a*(delGam[0]**f), a*(delGam[end]**f)], 'k--')
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel(r'$|\Delta \gamma|$', fontsize=15)
    plt.ylabel('K', fontsize=15)
    plt.legend()

In [16]:
## F PLOT WITH SLIDERS -------------------------------- #
critStrain = 0.1
interactive_plot_F = interactive( update_f,
    critStrain = widgets.FloatSlider(min=critStrain-0.05, max=critStrain+0.05, step=1e-3, value=critStrain, readout_format='.3f', description="gamC"),
    end = widgets.IntSlider(min=25, max=50, step=1, value=20, description="end"),
    a = widgets.FloatSlider(min=0.1, max=1.0, step=1e-3, value=0.3, readout_format='.2f', description="a"),
    f = widgets.FloatSlider(min=0.6, max=0.8, step=1e-2, value=0.66, readout_format='.2f', description="f"))

display(interactive_plot_F)

interactive(children=(FloatSlider(value=0.1, description='gamC', max=0.15000000000000002, min=0.05, readout_fo…

In [9]:
## NON-AFFINITY SLIDER FUNCTION -------------------------------- #
Ws = [20,40,80,160,200,500]
def update_NA(deform, critStrain):
    plt.figure()

    for count, W in enumerate(Ws):
        data0 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V0_r1.dat')
        data1 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V0_r-1.dat')
        data2 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V1_r1.dat')
        data3 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V1_r-1.dat')

        nonaff = 0.25 * ( data0[:,1] + data1[:,1] + data2[:,1] + data3[:,1] )
        plt.axvline( x = critStrain, c='k', ls='--' )
        plt.scatter( data0[:,0], nonaff, s=25, label=str(W) )

    plt.xlabel(r'$\Delta \gamma$', fontsize=20)
    plt.ylabel(r'$\delta \Gamma$', fontsize=20)
    plt.xscale('log')
    plt.yscale('log')
    plt.legend(title='$W$')

In [10]:
## NON-AFFINITY DISPLAY -------------------------------- #
interactive_plot_NA = interactive( update_NA,
    critStrain = widgets.FloatSlider(min=0.05, max=0.15, step=1e-3, value=0.1, readout_format='.3f', description="gamC"),
    deform = widgets.FloatSlider(min=-0.1, max=0.03, step=1e-2, value=0.0, readout_format='.2f', description="E"))

display(interactive_plot_NA)

interactive(children=(FloatSlider(value=0.0, description='E', max=0.03, min=-0.1, step=0.01), FloatSlider(valu…

In [11]:
## FINITE SIZE SLIDER FUNCTION -------------------------------- #
#critStrain = [0.101,0.101,0.098,0.098,0.1]
critStrain = [0.09,0.110,0.105,0.115,0.115,0.115]
Ws = [20,40,80,160,200,500]
fs = [0.71,0.72,0.73,0.72,0.72,0.72]

def update_FS(deform):
    plt.figure()

    for count, W in enumerate(Ws):
        data0 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V0_r1.dat')
        data1 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V0_r-1.dat')
        data2 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V1_r1.dat')
        data3 = np.loadtxt(f'nonaffine/W{W}_Z3.20_K6_E{deform:.2f}_V1_r-1.dat')

        nonaff = 0.25 * ( data0[:,1] + data1[:,1] + data2[:,1] + data3[:,1] )
        nu = 0.5*fs[count] + 1

        plt.scatter( (data0[:,0]-critStrain[count])*(W**(1/nu)), nonaff*(W**(-1.5/nu)), s=25, label=str(W) )

        plt.xlabel(r'$\Delta \gamma W^{1/\nu}$', fontsize=20)
        plt.ylabel(r'$\delta \Gamma W^{(f-\phi)/\nu}$', fontsize=20)
        plt.xlim([-5,20])
        plt.ylim([0.005,10])
        #plt.xscale('log')
        plt.yscale('log')
        plt.legend(title='$W$')

In [12]:
## FINITE SIZE DISPLAY -------------------------------- #
interactive_plot_FS = interactive( update_FS,
    deform = widgets.FloatSlider(min=-0.1, max=0.03, step=1e-2, value=0.0, readout_format='.2f', description="E"))

display(interactive_plot_FS)

interactive(children=(FloatSlider(value=0.0, description='E', max=0.03, min=-0.1, step=0.01), Output()), _dom_…